In [1]:
!pip install pyngrok nest-asyncio -q

In [2]:
import nest_asyncio
import uvicorn
import threading
from pyngrok import ngrok
import re, json, numpy as np, tensorflow as tf
from tensorflow.keras.layers import Layer, TextVectorization
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from contextlib import asynccontextmanager

nest_asyncio.apply()

class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="attention_weight",
                                 shape=(input_shape[-1], 1),
                                 initializer="random_normal", trainable=True)
        self.b = self.add_weight(name="attention_bias",
                                 shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, inputs):
        score = tf.nn.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        return tf.reduce_sum(attention_weights * inputs, axis=1)

try:
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    _factory  = StopWordRemoverFactory()
    _all_sw   = set(_factory.get_stop_words())
    _keep     = {
        'tidak','tak','bukan','tanpa','jangan','belum',
        'sangat','sekali','banget','amat','terlalu','lebih',
        'kurang','cukup','agak','lumayan',
        'baik','bagus','buruk','jelek','enak','susah','mudah',
        'senang','puas','kecewa','nyaman','aman','bahaya',
        'seru','membosankan','menarik','membingungkan',
        'cepat','lambat','lancar','error','gagal','berhasil',
        'perlu','harus','bisa','mampu','sulit',
        'semoga','harap','tolong','mohon',
    }
    _stopwords = _all_sw - _keep
except ImportError:
    _stopwords = set()

_slang_dict = {
    "gk":"tidak","ga":"tidak","bgt":"banget","yg":"yang",
    "dgn":"dengan","utk":"untuk","krn":"karena","sdh":"sudah",
    "blm":"belum","jg":"juga","lg":"lagi","tp":"tapi",
    "gak":"tidak","nggak":"tidak","gpp":"tidak apa",
    "mksh":"terima kasih","tks":"terima kasih",
}

def _preprocess(text: str) -> str:
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = ' '.join([_slang_dict.get(w, w) for w in text.split()])
    text = ' '.join([w for w in text.split() if w not in _stopwords])
    return text

model = tf.keras.models.load_model(
    "final_model.keras",
    custom_objects={"AttentionLayer": AttentionLayer}
)
print("Model loaded")

with open("vectorizer_config.json", "r") as f:
    vec_data = json.load(f)
vectorizer = TextVectorization.from_config(vec_data["config"])
vectorizer.set_vocabulary(vec_data["vocabulary"])
print(f"Vocabulary size: {len(vec_data['vocabulary'])}")

app = FastAPI(
    title="Sentiment Analysis API",
    description="Prediksi sentimen teks bahasa Indonesia — PKKMB UI",
    version="1.0.0"
)

class TextInput(BaseModel):
    text: str

class BatchInput(BaseModel):
    texts: list[str]

def _predict(text: str) -> dict:
    cleaned    = _preprocess(text)
    vectorized = vectorizer([cleaned])
    proba      = model.predict(vectorized, verbose=0)[0]
    label_idx  = int(np.argmax(proba))
    mapping    = {0: "negatif", 1: "netral", 2: "positif"}
    return {
        "input_asli"  : text,
        "teks_bersih" : cleaned,
        "sentimen"    : mapping[label_idx],
        "confidence"  : f"{proba[label_idx]*100:.2f}%",
        "probabilitas": {
            "negatif": f"{proba[0]*100:.2f}%",
            "netral" : f"{proba[1]*100:.2f}%",
            "positif": f"{proba[2]*100:.2f}%",
        }
    }

@app.get("/")
def root():
    return {"message": "Sentiment Analysis API is running", "status": "ok"}

@app.get("/health")
def health():
    return {"status": "healthy", "model_loaded": model is not None,
            "classes": ["negatif", "netral", "positif"]}

@app.post("/predict")
def predict(payload: TextInput):
    if not payload.text.strip():
        raise HTTPException(status_code=400, detail="Teks tidak boleh kosong.")
    return _predict(payload.text)

@app.post("/predict/batch")
def predict_batch(payload: BatchInput):
    if not payload.texts:
        raise HTTPException(status_code=400, detail="List teks kosong.")
    if len(payload.texts) > 50:
        raise HTTPException(status_code=400, detail="Maksimal 50 teks per request.")
    return {"results": [_predict(t) for t in payload.texts], "total": len(payload.texts)}

ngrok.set_auth_token("ISI_TOKEN_ANDA")

ngrok.kill()

public_url = ngrok.connect(8000)
print(f"\n Public URL : {public_url}")
print(f"   Swagger    : {public_url}/docs")
print(f"   Health     : {public_url}/health\n")

thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000),
    daemon=True
)
thread.start()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 20 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model loaded
Vocabulary size: 15000

 Public URL : NgrokTunnel: "https://shininess-yeah-ignore.ngrok-free.dev" -> "http://localhost:8000"
   Swagger    : NgrokTunnel: "https://shininess-yeah-ignore.ngrok-free.dev" -> "http://localhost:8000"/docs
   Health     : NgrokTunnel: "https://shininess-yeah-ignore.ngrok-free.dev" -> "http://localhost:8000"/health

